In [1]:
from datasets import load_dataset

In [2]:
dataset = load_dataset("Vikhrmodels/ToneBooks")# , "train")
dataset

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'text_description', 'voice_name'],
        num_rows: 91976
    })
    validation: Dataset({
        features: ['audio', 'text', 'text_description', 'voice_name'],
        num_rows: 4841
    })
})

In [3]:
dataset["train"][0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x780e9e531a90>,
 'text': 'Вот только они совсем не радовали, а, напротив, потрясали и ужасали.',
 'text_description': 'Accent: Стандартный американский английский Среднего Запада. Tone: Тёплый и ясный. Phrasing: Чёткое выделение смысловых частей с умеренными паузами для ритмического акцента. Pauses: Незначительные паузы для ритмического подчёркивания. Pronunciation: Точная и чёткая артикуляция без перебиваний или запинок. Emotion: Любознательный и воодушевляющий.',
 'voice_name': 'aleksandr_kotov'}

In [1]:
from datasets import load_from_disk
dataset = load_from_disk("../data/test_processed")


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'text_description', 'voice_name', 'phoneme_timestamps', 'processed', 'num_phonemes'],
        num_rows: 2
    })
    validation: Dataset({
        features: ['audio', 'text', 'text_description', 'voice_name', 'phoneme_timestamps', 'processed', 'num_phonemes'],
        num_rows: 2
    })
})

In [3]:
dataset["train"][0]

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7107d07c0aa0>,
 'text': 'Вот только они совсем не радовали, а, напротив, потрясали и ужасали.',
 'text_description': 'Accent: Стандартный американский английский Среднего Запада. Tone: Тёплый и ясный. Phrasing: Чёткое выделение смысловых частей с умеренными паузами для ритмического акцента. Pauses: Незначительные паузы для ритмического подчёркивания. Pronunciation: Точная и чёткая артикуляция без перебиваний или запинок. Emotion: Любознательный и воодушевляющий.',
 'voice_name': 'aleksandr_kotov',
 'phoneme_timestamps': [{'end': 0.38999998569488525,
   'phoneme': 'v',
   'start': 0.33000001311302185},
  {'end': 0.46000000834465027, 'phoneme': 'o', 'start': 0.38999998569488525},
  {'end': 0.5600000023841858, 'phoneme': 't̪', 'start': 0.46000000834465027},
  {'end': 0.6100000143051147, 'phoneme': 't̪', 'start': 0.5600000023841858},
  {'end': 0.6600000262260437, 'phoneme': 'o', 'start': 0.6100000143051147},
  {'end': 0.70999997854

In [4]:
# Function to extract hard "С" (s̪) timestamps
def extract_hard_s_timestamps(example):
    """
    Extract timestamps for all hard 'С' sounds (s̪ phoneme) from phoneme_timestamps.
    Returns a list of tuples (start, end) for each hard 'С' occurrence.
    """
    if 'phoneme_timestamps' not in example or example['phoneme_timestamps'] is None:
        return {'hard_s_timestamps': []}
    
    hard_s_timestamps = []
    for phoneme_info in example['phoneme_timestamps']:
        # Hard 'С' is represented as 's̪' in IPA (with diacritic)
        if phoneme_info['phoneme'] == 's̪':
            hard_s_timestamps.append((phoneme_info['start'], phoneme_info['end']))
    
    return {'hard_s_timestamps': hard_s_timestamps}

# Apply the function to the dataset
dataset = dataset.map(extract_hard_s_timestamps)

In [5]:
# Display example with hard 'С' timestamps
example = dataset["train"][0]
print(f"Text: {example['text']}")
print(f"\nNumber of hard 'С' sounds found: {len(example['hard_s_timestamps'])}")
print(f"\nHard 'С' timestamps: {example['hard_s_timestamps']}")

Text: Вот только они совсем не радовали, а, напротив, потрясали и ужасали.

Number of hard 'С' sounds found: 3

Hard 'С' timestamps: [[1.0499999523162842, 1.149999976158142], [3.619999885559082, 3.7899999618530273], [4.599999904632568, 4.75]]


In [7]:
import numpy as np
from IPython.display import Audio, display

# Load audio from the example (AudioDecoder supports dict-style access)
audio_array = example['audio']['array']
sample_rate = example['audio']['sampling_rate']

# Convert to numpy if needed
if not isinstance(audio_array, np.ndarray):
    audio_array = np.array(audio_array)

print(f"Sample rate: {sample_rate} Hz")
print(f"Total audio duration: {len(audio_array) / sample_rate:.2f} seconds")
print(f"\nDisplaying {len(example['hard_s_timestamps'])} hard 'С' (s̪) sound segments:\n")

# Display each hard 's' segment
for i, (start, end) in enumerate(example['hard_s_timestamps'], 1):
    # Convert time to sample indices
    start_sample = int(start * sample_rate)
    end_sample = int(end * sample_rate)
    
    # Extract audio segment
    segment = audio_array[start_sample:end_sample]
    
    print(f"Hard 'С' #{i}: [{start:.3f}s - {end:.3f}s] (duration: {end - start:.3f}s)")
    display(Audio(segment, rate=sample_rate))
    print()

Sample rate: 44100 Hz
Total audio duration: 5.26 seconds

Displaying 3 hard 'С' (s̪) sound segments:

Hard 'С' #1: [1.050s - 1.150s] (duration: 0.100s)



Hard 'С' #2: [3.620s - 3.790s] (duration: 0.170s)



Hard 'С' #3: [4.600s - 4.750s] (duration: 0.150s)


In [8]:
example

{'audio': <datasets.features._torchcodec.AudioDecoder at 0x7107cfec06e0>,
 'text': 'Вот только они совсем не радовали, а, напротив, потрясали и ужасали.',
 'text_description': 'Accent: Стандартный американский английский Среднего Запада. Tone: Тёплый и ясный. Phrasing: Чёткое выделение смысловых частей с умеренными паузами для ритмического акцента. Pauses: Незначительные паузы для ритмического подчёркивания. Pronunciation: Точная и чёткая артикуляция без перебиваний или запинок. Emotion: Любознательный и воодушевляющий.',
 'voice_name': 'aleksandr_kotov',
 'phoneme_timestamps': [{'end': 0.38999998569488525,
   'phoneme': 'v',
   'start': 0.33000001311302185},
  {'end': 0.46000000834465027, 'phoneme': 'o', 'start': 0.38999998569488525},
  {'end': 0.5600000023841858, 'phoneme': 't̪', 'start': 0.46000000834465027},
  {'end': 0.6100000143051147, 'phoneme': 't̪', 'start': 0.5600000023841858},
  {'end': 0.6600000262260437, 'phoneme': 'o', 'start': 0.6100000143051147},
  {'end': 0.70999997854